# 分词 - 练习

关于 [分词 (tokenization) 视频](https://www.youtube.com/watch?v=zduSFxRajkE) 中练习的笔记。<br>
改编自 [github.com/karpathy/minbpe/exercise.md](https://github.com/karpathy/minbpe/blob/master/exercise.md)。

1. 观看 YouTube 上的 [分词视频](https://www.youtube.com/watch?v=zduSFxRajkE)
2. 回来完成这些练习以提升自己 :)

**构建你自己的 GPT-4 分词器 (Tokenizer)！**

### 第 1 步

编写 `BasicTokenizer` 类，包含以下三个核心函数：

- `def train(self, text, vocab_size, verbose=False)`
- `def encode(self, text)`
- `def decode(self, ids)`

使用你喜欢的任意文本来训练你的分词器，并可视化合并后的 token。<br>
**它们看起来合理吗？**<br><br>
你可能希望使用的一个默认测试是文本文件 `taylorswift.txt`。

In [2]:
# TODO

### 第 2 步

将你的 `BasicTokenizer` 转换为 `RegexTokenizer`，它接受一个正则表达式模式并完全按照 GPT-4 的方式分割文本。<br>
像之前一样分别处理各部分，然后拼接结果。<br>
重新训练你的分词器并比较前后的结果。<br><br>
你应该看到，现在不会再有跨越类别的 token（数字、字母、标点、多个空格）。<br><br><br>
使用 GPT-4 的模式：

```
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```

In [ ]:
# TODO

### 第 3 步

现在你已经准备好加载 GPT-4 分词器的合并规则，并证明你的分词器在 `encode` 和 `decode` 上产生了与 [tiktoken](https://github.com/openai/tiktoken) 相同的结果。

```
# match this
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (\uc548\ub155\ud558\uc138\uc694!) lol123 \U0001f609")
text = enc.decode(ids) # get the same text back
```

不幸的是，你会遇到两个问题：

1. 从 GPT-4 分词器中恢复原始合并 (merges) 并非易事。你可以轻松恢复我们这里称为 `vocab` 的部分，也就是他们称为并存储在 `enc._mergeable_ranks` 下的内容。你可以自由复制粘贴 [`minbpe/gpt4.py`](https://github.com/karpathy/minbpe/blob/master/minbpe/gpt4.py) 中的 `recover_merges` 函数，它接受这些排名并返回原始合并。如果你想了解这个函数的工作原理，请阅读 [这篇](https://github.com/openai/tiktoken/issues/60) 和 [这篇](https://github.com/karpathy/minbpe/issues/11#issuecomment-1950805306)。基本上，在某些条件下，只需要存储父节点（及其排名）就足够了，而无需保留哪些子节点合并到任何父节点的精确细节。
2. 其次，GPT-4 分词器出于某种原因对其原始字节进行了排列。它将此排列存储在可合并排名 (mergeable ranks) 的前 $256$ 个元素中，因此你可以相对简单地恢复此字节洗牌，如 `byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}`。在你的 encode 和 decode 中，你都必须相应地对字节进行洗牌。

In [ ]:
# TODO

### 第 4 步

*（可选，令人烦恼，用途不明显）*<br><br>
添加处理特殊 token 的能力。<br>
这样你就能在存在特殊 token 时匹配 tiktoken 的输出，例如：

```
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("\nhello world", allowed_special="all")
```

如果不指定 `allowed_special`，tiktoken 会报错。

In [ ]:
# TODO

### 第 5 步

**如果你已经走到了这一步，那你现在已经是 LLM 分词的专家了！**<br><br>
遗憾的是，你还没有*完全*完成，因为 OpenAI 之外的许多 LLM（例如 Llama、Mistral）使用 [sentencepiece](https://github.com/google/sentencepiece) 而不是 tiktoken。<br>
主要区别在于 sentencepiece 直接在 Unicode 码点 (code points) 上运行 BPE，而不是在 UTF-8 编码的字节上运行。<br><br>
你可以自行探索 sentencepiece（祝你好运，它不太好看），<br>
如果你真的有时间且愿意承受这份痛苦，<br>
可以作为延伸目标，将你的 BPE 重写为在 Unicode 码点上运行，并匹配 Llama 2 分词器。

In [ ]:
# TODO